In [1]:
import numpy as np
import nibabel as nib
import pandas as pd

Command to compute geodesic distance between each node in fsaverage5 surface (10242 nodes) and all other nodes:

```bash
for hemi in lh rh; do
    parallel \
        --results "/RAID1/jupytertmp/mcm/data/fsa5-geodistance/nodes-dist/${hemi}_pial-node{}_gdist.txt" \
        --verbose \
        --jobs 20 \
        --delay 0.5 \
        SurfDist \
            -i "/RAID1/jupytertmp/mcm/data/external/fsaverage5/gifti/${hemi}_pial.gii" \
            -input "/RAID1/jupytertmp/mcm/data/fsa5-geodistance/nodes.1D" \
            -from_node {} ::: $(seq 0 10241)
done
```

In [2]:
resolution = 'fsaverage5'

annot_lh = f'/RAID1/jupytertmp/mcm/data/external/schaefer/{resolution}/lh.Schaefer2018_400Parcels_7Networks_order.annot'
annot_rh = f'/RAID1/jupytertmp/mcm/data/external/schaefer/{resolution}/rh.Schaefer2018_400Parcels_7Networks_order.annot'

labels = {
    'lh': nib.freesurfer.read_annot(annot_lh)[0].astype(int),
    'rh': nib.freesurfer.read_annot(annot_rh)[0].astype(int)
}

FileNotFoundError: [Errno 2] No such file or directory: '/RAID1/jupytertmp/mcm/data/external/schaefer/fsaverage5/lh.Schaefer2018_1000Parcels_7Networks_order.annot'

In [3]:
nodes_roi = {'lh': [], 'rh': []}

for hemi in ['lh', 'rh']:
    for node in range(10242):

        filename = f'/RAID1/jupytertmp/mcm/data/fsa5-geodistance/nodes-dist/{hemi}_pial-node{node}_gdist.txt'
        dist = np.loadtxt(filename)[:, -1] # columns: node_from, node_to, distance. use the last
        assert dist.size == labels[hemi].size

        row = (
            pd.DataFrame(dict(roi=labels[hemi], dist=dist))
            .groupby('roi')
            .agg('min')
            .loc[lambda x: x.index != 0] # remove the 0 roi
            .to_numpy()
            .flatten()
        )
        nodes_roi[hemi].append(row)

        print(f'{hemi} {node}', end='\r')

In [4]:
for hemi in ['lh', 'rh']:
    (
        pd.DataFrame(nodes_roi[hemi])
        .assign(roi=labels[hemi])
        .groupby(by='roi')
        .agg('min')
        .loc[lambda x: x.index != 0]
        .reset_index()
        .drop('roi', axis=1)
        .pipe(lambda x: (x + x.T) / 2)
        .to_csv(
            f'/RAID1/jupytertmp/mcm/data/fsa5-geodistance/{hemi}.Schaefer2018_400Parcels_7Networks_geodistance.txt',
            header=False, index=False
            )
    )

In [5]:
hemi = 'lh'

D = (
    pd.DataFrame(nodes_roi[hemi])
    .assign(roi=labels[hemi])
    .groupby(by='roi')
    .agg('min')
    .loc[lambda x: x.index != 0]
    .reset_index()
    .drop('roi', axis=1)
    .pipe(lambda x: (x + x.T) / 2)
)

In [6]:
D

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
0,0.00,2.66,2.42,3.96,15.32,27.56,17.60,23.35,39.55,22.01,...,71.92,70.29,70.76,89.31,84.090,86.22,95.28,105.23,102.45,99.57
1,2.66,0.00,16.10,2.49,18.51,36.91,1.79,48.83,40.33,18.24,...,52.55,49.52,49.99,70.66,64.370,65.44,74.51,84.46,81.68,78.80
2,2.42,16.10,0.00,3.06,2.56,2.97,23.31,3.36,16.27,21.17,...,72.10,71.29,71.76,88.47,85.090,87.21,96.28,106.22,103.45,100.57
3,3.96,2.49,3.06,0.00,1.63,19.63,2.25,33.86,19.39,1.05,...,42.54,41.08,41.54,60.39,54.870,57.00,66.06,76.01,73.23,70.35
4,15.32,18.51,2.56,1.63,0.00,1.99,23.55,18.81,2.33,7.97,...,59.48,68.50,68.96,73.82,80.075,84.42,93.48,103.43,96.53,97.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,86.22,65.44,87.21,57.00,84.42,102.96,45.92,117.65,96.91,53.60,...,25.10,2.78,2.56,27.34,2.780,0.00,2.54,16.17,14.08,1.76
196,95.28,74.51,96.28,66.06,93.48,112.02,54.98,126.72,107.49,62.66,...,39.38,12.96,2.54,43.72,20.300,2.54,0.00,2.76,27.43,1.99
197,105.23,84.46,106.22,76.01,103.43,121.97,64.93,136.66,118.19,72.61,...,52.90,26.67,14.55,62.32,36.090,16.17,2.76,0.00,46.91,21.36
198,102.45,81.68,103.45,73.23,96.53,104.88,62.15,111.05,89.76,66.55,...,18.03,16.70,35.85,2.46,1.800,14.08,27.43,46.91,0.00,1.78
